In [1]:
import os
import cv2
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Conv3D, MaxPooling3D, Flatten, Dropout, Dense, Input, BatchNormalization,
    Reshape, MultiHeadAttention, Conv1D, LayerNormalization, GlobalAveragePooling1D,
    TimeDistributed, Conv2D, MaxPooling2D, Activation, GlobalAveragePooling2D, LSTM, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, top_k_accuracy_score
from tensorflow.keras import mixed_precision

# 1.1 Reproducibility and Environment Setup
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU Setup
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Using GPU: {gpus[0]}")
    except RuntimeError as e:
        print(e)

# Note: Mixed precision is disabled for baseline reproducibility 
# to prevent activation overflow (NaNs) in un-normalized architectures.

# Constants
IMG_SIZE = 64
MAX_FRAMES = 20
BATCH_SIZE = 8
EPOCHS = 60
PATIENCE = 10

Using GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [2]:
import cv2
import os
import numpy as np

input_root = r"E:\PHD 2024\ISL-50"
output_root = r"E:\PHD 2024\ISL-50_Augmented"

os.makedirs(output_root, exist_ok=True)

def rotate_frame(frame, angle):
    h, w = frame.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(frame, M, (w, h))

def change_brightness(frame, factor):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    hsv = hsv.astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * factor, 0, 255)
    hsv = hsv.astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

for video in os.listdir(input_root):

    if not video.lower().endswith((".mp4", ".avi", ".mov")):
        continue

    video_path = os.path.join(input_root, video)

    augmentations = {
        "orig": None,
        "left": 5,
        "right": -5,
        "bright": 1.3
    }

    for aug_name, param in augmentations.items():

        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            print("Cannot open:", video_path)
            continue

        fps = cap.get(cv2.CAP_PROP_FPS)

        if fps == 0:
            fps = 30

        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        base_name = os.path.splitext(video)[0]

        out_path = os.path.join(
            output_root,
            f"{base_name}_{aug_name}.mp4"
        )

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')

        out = cv2.VideoWriter(
            out_path,
            fourcc,
            fps,
            (width, height)
        )

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            if aug_name == "left":
                frame = rotate_frame(frame, 5)

            elif aug_name == "right":
                frame = rotate_frame(frame, -5)

            elif aug_name == "bright":
                frame = change_brightness(frame, 1.3)

            out.write(frame)

        cap.release()
        out.release()

        print("Saved:", out_path)

print("Augmentation completed")

Augmentation completed


In [3]:
import os

input_root = r"D:\LOSO DATASET"

total_videos = 0

for word in os.listdir(input_root):

    word_path = os.path.join(input_root, word)

    if os.path.isdir(word_path):

        count = len([
            f for f in os.listdir(word_path)
            if f.lower().endswith((".mp4", ".avi", ".mov"))
        ])

        print(word, "=", count)
        total_videos += count

print("\nTotal videos =", total_videos)

s1 = 0
s2 = 0
s3 = 0
s4 = 0

Total videos = 0


In [4]:
import os

input_root = r"D:\LOSO DATASET"

total_videos = 0

for signer in os.listdir(input_root):

    signer_path = os.path.join(input_root, signer)

    if not os.path.isdir(signer_path):
        continue

    signer_count = 0

    for word in os.listdir(signer_path):

        word_path = os.path.join(signer_path, word)

        if not os.path.isdir(word_path):
            continue

        count = len([
            f for f in os.listdir(word_path)
            if f.lower().endswith((".mp4", ".avi", ".mov"))
        ])

        signer_count += count

    print(f"{signer} = {signer_count}")

    total_videos += signer_count

print("\nTotal videos =", total_videos)

s1 = 1304
s2 = 1108
s3 = 888
s4 = 740

Total videos = 4040


In [5]:
import os

input_root = r"D:\LOSO DATASET"

for signer in os.listdir(input_root):

    signer_path = os.path.join(input_root, signer)

    if not os.path.isdir(signer_path):
        continue

    print(f"\n===== {signer} =====")

    total = 0

    for word in sorted(os.listdir(signer_path)):

        word_path = os.path.join(signer_path, word)

        if os.path.isdir(word_path):

            count = len([
                f for f in os.listdir(word_path)
                if f.endswith(".mp4")
            ])

            print(f"{word}: {count}")

            total += count

    print(f"Total = {total}")


===== s1 =====
BLAZER: 12
BRUSHING: 28
Document folder: 24
Family: 24
Father: 24
Husband: 24
Mother: 20
Sister: 20
TV: 24
Tree: 24
aunt: 20
baby_boy: 28
baby_girl: 32
bag: 32
brother: 24
bye: 24
clean: 20
doctor: 24
drink: 24
eat: 36
exam: 40
exam result: 36
feel: 20
friend: 20
grandfather: 24
grandmother: 28
namaste: 28
neighbour: 28
nurse: 20
paper: 36
pencil: 36
please: 24
sit: 20
sleep: 20
sorry: 24
sports or play: 24
stand up: 28
student: 36
table: 40
talking: 28
teacher: 24
thank you: 24
time: 20
topic: 32
uncle: 24
visit: 20
walking: 28
wife: 24
work: 36
writing: 24
Total = 1304

===== s2 =====
BLAZER: 20
BRUSHING: 16
Document folder: 24
Family: 20
Father: 20
Husband: 20
Mother: 20
Sister: 20
TV: 24
Tree: 24
aunt: 20
baby_boy: 20
baby_girl: 20
bag: 20
brother: 20
bye: 28
clean: 24
doctor: 20
drink: 20
eat: 28
exam: 24
exam result: 20
feel: 24
friend: 20
grandfather: 20
grandmother: 20
namaste: 24
neighbour: 24
nurse: 24
paper: 28
pencil: 20
please: 24
sit: 16
sleep: 28
sorry: 2

In [6]:
import os
import numpy as np

DATA_DIR = r"D:\LOSO DATASET"

def get_video_paths_loso(data_dir):

    video_paths = []
    labels = []
    groups = []

    # Signer folders (S1, S2, S3, S4)
    signer_folders = sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ])

    # Word folders
    class_names = sorted(os.listdir(os.path.join(data_dir, signer_folders[0])))

    label_map = {cls: idx for idx, cls in enumerate(class_names)}

    for signer in signer_folders:

        signer_path = os.path.join(data_dir, signer)

        for cls in class_names:

            class_path = os.path.join(signer_path, cls)

            if not os.path.exists(class_path):
                continue

            for file in os.listdir(class_path):

                if file.lower().endswith(".mp4"):

                    video_paths.append(os.path.join(class_path, file))
                    labels.append(label_map[cls])
                    groups.append(signer)

    return (
        np.array(video_paths),
        np.array(labels),
        np.array(groups),
        class_names
    )


video_paths, labels, groups, class_names = get_video_paths_loso(DATA_DIR)

print("Total videos :", len(video_paths))
print("Classes      :", len(class_names))
print("Signers      :", np.unique(groups))

Total videos : 4040
Classes      : 50
Signers      : ['s1' 's2' 's3' 's4']


In [7]:
from sklearn.model_selection import LeaveOneGroupOut

logo = LeaveOneGroupOut()

for fold, (train_idx, test_idx) in enumerate(
        logo.split(video_paths, labels, groups), start=1):

    print("=" * 60)
    print(f"Fold {fold}")

    print("Training Signers :", np.unique(groups[train_idx]))
    print("Testing Signer   :", np.unique(groups[test_idx]))

    print("Training Samples :", len(train_idx))
    print("Testing Samples  :", len(test_idx))

Fold 1
Training Signers : ['s2' 's3' 's4']
Testing Signer   : ['s1']
Training Samples : 2736
Testing Samples  : 1304
Fold 2
Training Signers : ['s1' 's3' 's4']
Testing Signer   : ['s2']
Training Samples : 2932
Testing Samples  : 1108
Fold 3
Training Signers : ['s1' 's2' 's4']
Testing Signer   : ['s3']
Training Samples : 3152
Testing Samples  : 888
Fold 4
Training Signers : ['s1' 's2' 's3']
Testing Signer   : ['s4']
Training Samples : 3300
Testing Samples  : 740


In [8]:
# ============================
# 2.1 LOSO Dataset Discovery
# ============================

from sklearn.model_selection import LeaveOneGroupOut

DATA_DIR = r"D:\LOSO DATASET"

def get_video_paths_loso(data_dir):

    video_paths = []
    labels = []
    groups = []

    signer_folders = sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ])

    class_names = sorted([
        d for d in os.listdir(os.path.join(data_dir, signer_folders[0]))
        if os.path.isdir(os.path.join(data_dir, signer_folders[0], d))
    ])

    label_map = {cls:i for i,cls in enumerate(class_names)}

    for signer in signer_folders:

        signer_path = os.path.join(data_dir, signer)

        for cls in class_names:

            class_path = os.path.join(signer_path, cls)

            if not os.path.exists(class_path):
                continue

            for file in os.listdir(class_path):

                if file.lower().endswith(".mp4"):

                    video_paths.append(os.path.join(class_path,file))
                    labels.append(label_map[cls])
                    groups.append(signer)

    return np.array(video_paths), np.array(labels), np.array(groups), class_names


video_paths, labels, groups, class_names = get_video_paths_loso(DATA_DIR)

num_classes = len(class_names)

print("Total Videos :",len(video_paths))
print("Classes      :",num_classes)
print("Signers      :",np.unique(groups))

Total Videos : 4040
Classes      : 50
Signers      : ['s1' 's2' 's3' 's4']


In [11]:
# 3.1 Video Loading, Validation, and Dataset Assembly
def load_video_frames(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened(): return None, "open_failed", path
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0: return None, "empty_video", path
    indices = set(np.linspace(0, total_frames - 1, MAX_FRAMES, dtype=int))
    frame_count, collected_frames = 0, []
    while True:
        ret, frame = cap.read()
        if not ret: break
        if frame_count in indices:
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            collected_frames.append(frame)
        frame_count += 1
        if len(collected_frames) >= MAX_FRAMES: break
    cap.release()
    if len(collected_frames) == 0: return None, "no_decoded_frames", path
    video = np.array(collected_frames)
    if len(video) < MAX_FRAMES:
        padding = np.tile(video[-1], (MAX_FRAMES - len(video), 1, 1, 1))
        video = np.concatenate((video, padding), axis=0)
    video = video[:MAX_FRAMES].astype("float32") / 255.0
    if float(np.std(video)) < 1e-4 or float(np.max(video)) < 1e-3: return video, "low_variance", path
    return video, "ok", path

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
def load_dataset_in_parallel(paths, labels, desc="Loading"):
    max_workers = min(16, (os.cpu_count() or 4) * 2)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(load_video_frames, paths), total=len(paths), desc=desc))
    X_data, y_data = [], []
    skipped = {"open_failed": 0, "empty_video": 0, "no_decoded_frames": 0}
    for idx, (video, status, path) in enumerate(results):
        if status in skipped:
            skipped[status] += 1
            continue
        X_data.append(video)
        y_data.append(labels[idx])
    print(f"{desc}: kept {len(X_data)}/{len(paths)} videos; skipped {skipped}")
    return np.array(X_data), np.array(y_data)


def augment_video(video, label):
    video = tf.image.random_brightness(video, max_delta=0.04)
    video = tf.image.random_contrast(video, lower=0.95, upper=1.05)
    video = tf.image.resize_with_crop_or_pad(video, IMG_SIZE + 2, IMG_SIZE + 2)
    video = tf.image.random_crop(video, size=(MAX_FRAMES, IMG_SIZE, IMG_SIZE, 3))
    noise = tf.random.normal(tf.shape(video), mean=0.0, stddev=0.01, dtype=tf.float32)
    video = tf.clip_by_value(video + noise, 0.0, 1.0)
    return video, label

def create_dataset_v3(X, y, shuffle=False, augment=False, batch_size=BATCH_SIZE):
    y = np.asarray(y, dtype=np.int32)
    def sample_gen():
        for i in range(len(X)): yield X[i], y[i]
    ds = tf.data.Dataset.from_generator(
        sample_gen,
        output_signature=(
            tf.TensorSpec(shape=(MAX_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32),
        ),
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(X), 2048), seed=SEED, reshuffle_each_iteration=True)
    if augment:
        ds = ds.map(augment_video, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [12]:
from sklearn.model_selection import LeaveOneGroupOut, train_test_split

logo = LeaveOneGroupOut()

fold_results = []

for fold, (train_idx, test_idx) in enumerate(
        logo.split(video_paths, labels, groups), start=1):

    print("\n" + "="*70)
    print(f"LOSO Fold {fold}")

    # Split by signer
    train_paths = video_paths[train_idx]
    test_paths = video_paths[test_idx]

    y_train = labels[train_idx]
    y_test = labels[test_idx]

    # Validation split from training signers only
    train_paths, val_paths, y_train, y_val = train_test_split(
        train_paths,
        y_train,
        test_size=0.10,
        stratify=y_train,
        random_state=42
    )

    # Load videos
    X_train_data, y_train_data = load_dataset_in_parallel(
        train_paths.tolist(),
        y_train.tolist(),
        desc=f"Fold {fold} Train"
    )

    X_val_data, y_val_data = load_dataset_in_parallel(
        val_paths.tolist(),
        y_val.tolist(),
        desc=f"Fold {fold} Val"
    )

    X_test_data, y_test_data = load_dataset_in_parallel(
        test_paths.tolist(),
        y_test.tolist(),
        desc=f"Fold {fold} Test"
    )

    print(f"Usable videos -> Train: {len(X_train_data)}, "
          f"Val: {len(X_val_data)}, Test: {len(X_test_data)}")

    # TensorFlow datasets
    train_ds = create_dataset_v3(
        X_train_data,
        y_train_data,
        shuffle=True,
        augment=True
    )

    val_ds = create_dataset_v3(
        X_val_data,
        y_val_data
    )

    test_ds = create_dataset_v3(
        X_test_data,
        y_test_data
    )

    


LOSO Fold 1


Fold 1 Train: 100%|██████████████████████████████████████████████████████████████| 2462/2462 [1:51:24<00:00,  2.72s/it]


Fold 1 Train: kept 2462/2462 videos; skipped {'open_failed': 0, 'empty_video': 0, 'no_decoded_frames': 0}


Fold 1 Val: 100%|████████████████████████████████████████████████████████████████████| 274/274 [12:00<00:00,  2.63s/it]


Fold 1 Val: kept 274/274 videos; skipped {'open_failed': 0, 'empty_video': 0, 'no_decoded_frames': 0}


Fold 1 Test: 100%|███████████████████████████████████████████████████████████████| 1304/1304 [1:03:01<00:00,  2.90s/it]


Fold 1 Test: kept 1304/1304 videos; skipped {'open_failed': 0, 'empty_video': 0, 'no_decoded_frames': 0}
Usable videos -> Train: 2462, Val: 274, Test: 1304


NameError: name 'build_model' is not defined

In [13]:
# 4.1 Hybrid VATN Model Architecture
def build_model(num_classes, input_shape):
    inputs = Input(shape=input_shape)
    
    # 1. Spatiotemporal Feature Extraction (TimeDistributed CNN)
    # Block 1
    x = TimeDistributed(Conv2D(32, (3, 3), padding='same', activation='relu'), name='block1_conv')(inputs)
    x = TimeDistributed(BatchNormalization(), name='block1_bn')(x)
    x = TimeDistributed(MaxPooling2D((2, 2)), name='block1_pool')(x)
    x = TimeDistributed(Dropout(0.15), name='block1_dropout')(x)
    
    # Block 2
    x = TimeDistributed(Conv2D(64, (3, 3), padding='same', activation='relu'), name='block2_conv')(x)
    x = TimeDistributed(BatchNormalization(), name='block2_bn')(x)
    x = TimeDistributed(MaxPooling2D((2, 2)), name='block2_pool')(x)
    x = TimeDistributed(Dropout(0.15), name='block2_dropout')(x)
    
    # Block 3
    x = TimeDistributed(Conv2D(128, (3, 3), padding='same', activation='relu'), name='block3_conv')(x)
    x = TimeDistributed(BatchNormalization(), name='block3_bn')(x)
    x = TimeDistributed(MaxPooling2D((2, 2)), name='block3_pool')(x)
    x = TimeDistributed(Dropout(0.20), name='block3_dropout')(x)
    
    # Flatten spatial dims: (Batch, Time, H*W*C)
    x = TimeDistributed(Flatten(), name='flatten')(x)
    
    # 2. Temporal Modeling (TCN / Conv1D)
    x = Conv1D(128, kernel_size=3, padding="causal", activation="relu", name='tcn1')(x)
    x = BatchNormalization(name='tcn1_bn')(x)
    x = Dropout(0.20, name='tcn1_dropout')(x)
    
    x = Conv1D(128, kernel_size=3, padding="causal", activation="relu", name='tcn2')(x)
    x = BatchNormalization(name='tcn2_bn')(x)
    x = Dropout(0.20, name='tcn2_dropout')(x)

    # 3. Global Temporal Attention
    attention = MultiHeadAttention(num_heads=4, key_dim=64, name='attention')(x, x)
    x = LayerNormalization(epsilon=1e-6, name='attn_norm')(attention + x)
    
    # Global Average Pooling over Time
    x = GlobalAveragePooling1D(name='global_avg_pool')(x)
    
    # Classification Head
    x = Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4), name='dense1')(x)
    x = Dropout(0.25, name='dense1_dropout')(x)
    
    # Output Layer
    outputs = Dense(num_classes, name='predictions')(x)
    outputs = Activation('softmax', dtype='float32', name='softmax')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs, name="Hybrid_VATN_ISL")
    return model

input_shape = (MAX_FRAMES, IMG_SIZE, IMG_SIZE, 3)

# Build once only to verify the architecture
temp_model = build_model(num_classes, input_shape)
temp_model.summary()
del temp_model

Model: "Hybrid_VATN_ISL"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 20, 64, 64,  0           []                               
                                 3)]                                                              
                                                                                                  
 block1_conv (TimeDistributed)  (None, 20, 64, 64,   896         ['input_1[0][0]']                
                                32)                                                               
                                                                                                  
 block1_bn (TimeDistributed)    (None, 20, 64, 64,   128         ['block1_conv[0][0]']            
                                32)                                                 

In [14]:
from sklearn.model_selection import LeaveOneGroupOut, train_test_split

logo = LeaveOneGroupOut()

fold_results = []

for fold, (train_idx, test_idx) in enumerate(
        logo.split(video_paths, labels, groups), start=1):

    print("\n" + "="*70)
    print(f"LOSO Fold {fold}")

    # Split by signer
    train_paths = video_paths[train_idx]
    test_paths = video_paths[test_idx]

    y_train = labels[train_idx]
    y_test = labels[test_idx]

    # Validation split from training signers only
    train_paths, val_paths, y_train, y_val = train_test_split(
        train_paths,
        y_train,
        test_size=0.10,
        stratify=y_train,
        random_state=42
    )

    # Load videos
    X_train_data, y_train_data = load_dataset_in_parallel(
        train_paths.tolist(),
        y_train.tolist(),
        desc=f"Fold {fold} Train"
    )

    X_val_data, y_val_data = load_dataset_in_parallel(
        val_paths.tolist(),
        y_val.tolist(),
        desc=f"Fold {fold} Val"
    )

    X_test_data, y_test_data = load_dataset_in_parallel(
        test_paths.tolist(),
        y_test.tolist(),
        desc=f"Fold {fold} Test"
    )

    print(f"Usable videos -> Train: {len(X_train_data)}, "
          f"Val: {len(X_val_data)}, Test: {len(X_test_data)}")

    # TensorFlow datasets
    train_ds = create_dataset_v3(
        X_train_data,
        y_train_data,
        shuffle=True,
        augment=True
    )

    val_ds = create_dataset_v3(
        X_val_data,
        y_val_data
    )

    test_ds = create_dataset_v3(
        X_test_data,
        y_test_data
    )



LOSO Fold 1


Fold 1 Train: 100%|██████████████████████████████████████████████████████████████| 2462/2462 [1:51:08<00:00,  2.71s/it]


Fold 1 Train: kept 2462/2462 videos; skipped {'open_failed': 0, 'empty_video': 0, 'no_decoded_frames': 0}


Fold 1 Val: 100%|████████████████████████████████████████████████████████████████████| 274/274 [12:34<00:00,  2.75s/it]


Fold 1 Val: kept 274/274 videos; skipped {'open_failed': 0, 'empty_video': 0, 'no_decoded_frames': 0}


...

Fold 3 Train: 100%|██████████████████████████████████████████████████████████████| 2836/2836 [2:10:53<00:00,  2.77s/it]


Fold 3 Train: kept 2836/2836 videos; skipped {'open_failed': 0, 'empty_video': 0, 'no_decoded_frames': 0}


Fold 3 Val: 100%|████████████████████████████████████████████████████████████████████| 316/316 [14:37<00:00,  2.78s/it]


Fold 3 Val: kept 316/316 videos; skipped {'open_failed': 0, 'empty_video': 0, 'no_decoded_frames': 0}


Fold 3 Test:  79%|█████████████████████████████████████████████████████              | 704/888 [32:43<10:53,  3.55s/it]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [15]:
# 4.1 Hybrid VATN Model Architecture
def build_model(num_classes, input_shape):
    inputs = Input(shape=input_shape)
    
    # 1. Spatiotemporal Feature Extraction (TimeDistributed CNN)
    # Block 1
    x = TimeDistributed(Conv2D(32, (3, 3), padding='same', activation='relu'), name='block1_conv')(inputs)
    x = TimeDistributed(BatchNormalization(), name='block1_bn')(x)
    x = TimeDistributed(MaxPooling2D((2, 2)), name='block1_pool')(x)
    x = TimeDistributed(Dropout(0.15), name='block1_dropout')(x)
    
    # Block 2
    x = TimeDistributed(Conv2D(64, (3, 3), padding='same', activation='relu'), name='block2_conv')(x)
    x = TimeDistributed(BatchNormalization(), name='block2_bn')(x)
    x = TimeDistributed(MaxPooling2D((2, 2)), name='block2_pool')(x)
    x = TimeDistributed(Dropout(0.15), name='block2_dropout')(x)
    
    # Block 3
    x = TimeDistributed(Conv2D(128, (3, 3), padding='same', activation='relu'), name='block3_conv')(x)
    x = TimeDistributed(BatchNormalization(), name='block3_bn')(x)
    x = TimeDistributed(MaxPooling2D((2, 2)), name='block3_pool')(x)
    x = TimeDistributed(Dropout(0.20), name='block3_dropout')(x)
    
    # Flatten spatial dims: (Batch, Time, H*W*C)
    x = TimeDistributed(Flatten(), name='flatten')(x)
    
    # 2. Temporal Modeling (TCN / Conv1D)
    x = Conv1D(128, kernel_size=3, padding="causal", activation="relu", name='tcn1')(x)
    x = BatchNormalization(name='tcn1_bn')(x)
    x = Dropout(0.20, name='tcn1_dropout')(x)
    
    x = Conv1D(128, kernel_size=3, padding="causal", activation="relu", name='tcn2')(x)
    x = BatchNormalization(name='tcn2_bn')(x)
    x = Dropout(0.20, name='tcn2_dropout')(x)

    # 3. Global Temporal Attention
    attention = MultiHeadAttention(num_heads=4, key_dim=64, name='attention')(x, x)
    x = LayerNormalization(epsilon=1e-6, name='attn_norm')(attention + x)
    
    # Global Average Pooling over Time
    x = GlobalAveragePooling1D(name='global_avg_pool')(x)
    
    # Classification Head
    x = Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4), name='dense1')(x)
    x = Dropout(0.25, name='dense1_dropout')(x)
    
    # Output Layer
    outputs = Dense(num_classes, name='predictions')(x)
    outputs = Activation('softmax', dtype='float32', name='softmax')(outputs)
    
    model = Model(inputs=inputs, outputs=outputs, name="Hybrid_VATN_ISL")
    return model

input_shape = (MAX_FRAMES, IMG_SIZE, IMG_SIZE, 3)

# Build once only to verify the architecture
temp_model = build_model(num_classes, input_shape)
temp_model.summary()
del temp_model

Model: "Hybrid_VATN_ISL"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 20, 64, 64,  0           []                               
                                 3)]                                                              
                                                                                                  
 block1_conv (TimeDistributed)  (None, 20, 64, 64,   896         ['input_2[0][0]']                
                                32)                                                               
                                                                                                  
 block1_bn (TimeDistributed)    (None, 20, 64, 64,   128         ['block1_conv[0][0]']            
                                32)                                                 

In [18]:
# Build model for current fold
input_shape = X_train_data.shape[1:]

model = build_model(num_classes, input_shape)

learning_rate = 3e-4
clip_val = 1.0

optimizer = tf.keras.optimizers.Adam(
    learning_rate=learning_rate,
    clipnorm=clip_val
)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top_3_acc"),
    ],
)

ckpt_path = f"VATN_ISL50_Fold{fold}.keras"

callbacks = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=max(2, PATIENCE // 2),
        min_lr=1e-6,
        verbose=1,
    ),
    ModelCheckpoint(
        ckpt_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
]

fit_kwargs = dict(
    x=train_ds,
    validation_data=val_ds,
    epochs=75,
    callbacks=callbacks,
    verbose=1,
)


history = model.fit(**fit_kwargs)

loss, acc, top3 = model.evaluate(test_ds, verbose=1)

print(f"\nFold {fold} Accuracy : {acc:.4f}")

fold_results.append(acc)

Epoch 1/75
    372/Unknown - 63s 103ms/step - loss: 3.7191 - accuracy: 0.0677 - top_3_acc: 0.1663
Epoch 1: val_accuracy improved from -inf to 0.03333, saving model to VATN_ISL50_Fold4.keras
372/372 [==============================] - 65s 108ms/step - loss: 3.7191 - accuracy: 0.0677 - top_3_acc: 0.1663 - val_loss: 3.9312 - val_accuracy: 0.0333 - val_top_3_acc: 0.0970 - lr: 3.0000e-04
Epoch 2/75
372/372 [==============================] - ETA: 0s - loss: 2.6047 - accuracy: 0.2623 - top_3_acc: 0.5310
Epoch 2: val_accuracy improved from 0.03333 to 0.33333, saving model to VATN_ISL50_Fold4.keras
372/372 [==============================] - 36s 93ms/step - loss: 2.6047 - accuracy: 0.2623 - top_3_acc: 0.5310 - val_loss: 2.1018 - val_accuracy: 0.3333 - val_top_3_acc: 0.6788 - lr: 3.0000e-04
Epoch 3/75
372/372 [==============================] - ETA: 0s - loss: 1.8193 - accuracy: 0.4535 - top_3_acc: 0.7394
Epoch 3: val_accuracy improved from 0.33333 to 0.60000, saving model to VATN_ISL50_Fold4.keras

In [19]:
# 6.1 Quantitative Evaluation (final holdout + publication metrics table)
print("[Section 6.1] Loading best checkpoint and running final test-set evaluation...")
if os.path.exists(ckpt_path): model.load_weights(ckpt_path)

import time
from sklearn.metrics import precision_recall_fscore_support, f1_score

test_loss, test_acc, test_top3 = model.evaluate(test_ds, verbose=1)
print(f"Final Test Loss:     {test_loss:.4f}")
print(f"Final Test Accuracy: {test_acc:.2%}")
print(f"Final Test Top-3:    {test_top3:.2%}")

start_t = time.perf_counter()
y_pred_probs_base = model.predict(test_ds, verbose=1)
per_sample_latency_ms = ((time.perf_counter() - start_t) / max(len(y_test_data), 1)) * 1000.0

y_pred_base = np.argmax(y_pred_probs_base, axis=1)
base_top1 = accuracy_score(y_test_data, y_pred_base)
try: base_top3 = top_k_accuracy_score(y_test_data, y_pred_probs_base, k=3)
except ValueError: base_top3 = 0.0

X_test_shift_p1 = np.roll(X_test_data, shift=1, axis=1)
X_test_shift_m1 = np.roll(X_test_data, shift=-1, axis=1)
test_ds_p1 = create_dataset_v3(X_test_shift_p1, y_test_data, shuffle=False)
test_ds_m1 = create_dataset_v3(X_test_shift_m1, y_test_data, shuffle=False)

y_pred_probs_p1 = model.predict(test_ds_p1, verbose=0)
y_pred_probs_m1 = model.predict(test_ds_m1, verbose=0)
y_pred_probs = (y_pred_probs_base + y_pred_probs_p1 + y_pred_probs_m1) / 3.0
y_pred = np.argmax(y_pred_probs, axis=1)

tta_top1 = accuracy_score(y_test_data, y_pred)
try: tta_top3 = top_k_accuracy_score(y_test_data, y_pred_probs, k=3)
except: tta_top3 = 0.0

macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_test_data, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_test_data, y_pred, average="micro", zero_division=0)
model_size_mb = os.path.getsize(ckpt_path) / (1024 ** 2) if os.path.exists(ckpt_path) else 0

print(f"Top-1 Accuracy (TTA): {tta_top1:.4f}")
print(f"Top-3 Accuracy (TTA): {tta_top3:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Latency (ms/sample): {per_sample_latency_ms:.3f}")

print(classification_report(y_test_data, y_pred, target_names=class_names, digits=2, zero_division=0))

[Section 6.1] Loading best checkpoint and running final test-set evaluation...
93/93 [==============================] - 3s 28ms/step - loss: 10.8572 - accuracy: 0.0108 - top_3_acc: 0.0649
Final Test Loss:     10.8572
Final Test Accuracy: 1.08%
Final Test Top-3:    6.49%
93/93 [==============================] - 3s 19ms/step
Top-1 Accuracy (TTA): 0.0108
Top-3 Accuracy (TTA): 0.0635
Macro F1: 0.0042
Latency (ms/sample): 4.256
                 precision    recall  f1-score   support

         BLAZER       0.00      0.00      0.00        20
       BRUSHING       0.00      0.00      0.00        20
Document folder       0.00      0.00      0.00        20
         Family       0.00      0.00      0.00        20
         Father       0.00      0.00      0.00        20
        Husband       0.00      0.00      0.00        16
         Mother       0.00      0.00      0.00         4
         Sister       0.00      0.00      0.00         4
             TV       0.05      0.38      0.08        16
  